In [0]:
# 1. DEFINICIÓN DE ENTORNOS Y CATÁLOGOS
catalog_name = "fintech_finpay"
schemas = ["default", "bronze", "silver", "gold", "observability"]

# Crear Catálogo Principal
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"USE CATALOG {catalog_name}")
print(f"✅ Catálogo '{catalog_name}' listo.")

# Crear Esquemas (Capas Medallion)
for schema in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema}")
    print(f"✅ Esquema '{schema}' creado exitosamente.")

✅ Catálogo 'fintech_finpay' listo.
✅ Esquema 'default' creado exitosamente.
✅ Esquema 'bronze' creado exitosamente.
✅ Esquema 'silver' creado exitosamente.
✅ Esquema 'gold' creado exitosamente.
✅ Esquema 'observability' creado exitosamente.


In [0]:
# 2. CREACIÓN DEL VOLUMEN DE ATERRIZAJE
volume_name = "vol_landing"
base_path = f"/Volumes/{catalog_name}/default/{volume_name}"
subdirs = ["transactions", "merchants", "users", "metadata", "metadata/schema", "metadata/checkpoints"]

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.default.{volume_name}")
print(f"✅ Volumen '{volume_name}' creado.")

# Crear carpetas internas usando dbutils
for subdir in subdirs:
    path = f"{base_path}/{subdir}"
    dbutils.fs.mkdirs(path)
    print(f"📁 Directorio listo: {path}")

✅ Volumen 'vol_landing' creado.
📁 Directorio listo: /Volumes/fintech_finpay/default/vol_landing/transactions
📁 Directorio listo: /Volumes/fintech_finpay/default/vol_landing/merchants
📁 Directorio listo: /Volumes/fintech_finpay/default/vol_landing/users
📁 Directorio listo: /Volumes/fintech_finpay/default/vol_landing/metadata
📁 Directorio listo: /Volumes/fintech_finpay/default/vol_landing/metadata/schema
📁 Directorio listo: /Volumes/fintech_finpay/default/vol_landing/metadata/checkpoints


In [0]:
# 3. RETO 1: METADATA-DRIVEN (Generar arquetipos automáticamente)
import json

archetypes = [
    {
        "source_name": "transactions",
        "source_path": f"{base_path}/transactions/",
        "file_format": "csv",
        "delimiter": ",",
        "header": True,
        "active": True
    },
    {
        "source_name": "merchants",
        "source_path": f"{base_path}/merchants/",
        "file_format": "json",
        "multiline": True,
        "active": True
    },
    {
        "source_name": "users",
        "source_path": f"{base_path}/users/",
        "file_format": "text",
        "delimiter": "|",
        "header": True,
        "active": True
    }
]

archetype_path = f"{base_path}/metadata/ingestion_archetypes.json"
dbutils.fs.put(archetype_path, json.dumps(archetypes, indent=2, ensure_ascii=False), overwrite=True)
print(f"✅ Archivo JSON de configuración creado en: {archetype_path}")

Wrote 598 bytes.
✅ Archivo JSON de configuración creado en: /Volumes/fintech_finpay/default/vol_landing/metadata/ingestion_archetypes.json


In [0]:
# 4. ASIGNACIÓN DE ROLES Y PERMISOS

import requests

catalog_name = "fintech_finpay"
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# --- 1. Crear grupos usando la API interna de Databricks ---
for grupo in ["ingenieria", "riesgo", "auditoria"]:
    url = f"https://{workspace_url}/api/2.0/preview/scim/v2/Groups"
    payload = {"schemas": ["urn:ietf:params:scim:schemas:core:2.0:Group"], "displayName": grupo}
    resp = requests.post(url, headers=headers, json=payload)

print("✅ Grupos locales validados en el Workspace.")

# --- 2. Asignar los permisos a los grupos (Con manejo de excepciones) ---
roles_config = {
    "ingenieria": ["default", "bronze", "silver", "gold", "observability"],
    "riesgo": ["silver", "gold"],
    "auditoria": ["gold", "observability"]
}

for rol, schemas_permitidos in roles_config.items():
    try:
        # Intentar dar permiso de uso del catálogo
        spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog_name} TO `{rol}`")
        
        for schema in schemas_permitidos:
            spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog_name}.{schema} TO `{rol}`")
            spark.sql(f"GRANT SELECT ON SCHEMA {catalog_name}.{schema} TO `{rol}`")
            
            # Ingeniería necesita permisos adicionales para crear y modificar tablas
            if rol == "ingenieria":
                spark.sql(f"GRANT CREATE TABLE ON SCHEMA {catalog_name}.{schema} TO `{rol}`")
                spark.sql(f"GRANT MODIFY ON SCHEMA {catalog_name}.{schema} TO `{rol}`")
                spark.sql(f"GRANT EXECUTE ON SCHEMA {catalog_name}.{schema} TO `{rol}`")
                
        print(f"🔐 Permisos aplicados exitosamente para el rol: {rol}")
        
    except Exception as e:
        # Si Unity Catalog bloquea el grupo por ser local, atrapamos el error para no romper el notebook
        if "PRINCIPAL_DOES_NOT_EXIST" in str(e):
            print(f"⚠️ Aviso: Se omite el GRANT para '{rol}' por restricción de Unity Catalog. Eres admin, el pipeline funcionará normal.")
        else:
            print(f"❌ Error inesperado con {rol}: {e}")

✅ Grupos locales validados en el Workspace.
⚠️ Aviso: Se omite el GRANT para 'ingenieria' por restricción de Unity Catalog. Eres admin, el pipeline funcionará normal.
⚠️ Aviso: Se omite el GRANT para 'riesgo' por restricción de Unity Catalog. Eres admin, el pipeline funcionará normal.
⚠️ Aviso: Se omite el GRANT para 'auditoria' por restricción de Unity Catalog. Eres admin, el pipeline funcionará normal.


In [0]:
# 5. SEGURIDAD DE DATOS: COLUMN MASKING Y ROW-LEVEL SECURITY
# Crear función para enmascarar datos PII (Solo ingeniería ve los datos reales)
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog_name}.silver.mask_pii(value STRING)
    RETURNS STRING
    RETURN CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('ingenieria') THEN value
        ELSE '***MASKED***'
    END
""")

# Crear función de seguridad a nivel de filas (Ingeniería y Riesgo pueden ver todas las filas)
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog_name}.silver.row_filter_users(country STRING)
    RETURNS BOOLEAN
    RETURN CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('ingenieria') THEN TRUE
        WHEN IS_ACCOUNT_GROUP_MEMBER('riesgo') THEN TRUE
        ELSE FALSE
    END
""")
print("🛡️ Funciones de seguridad (Masking y RLS) creadas.")

🛡️ Funciones de seguridad (Masking y RLS) creadas.


In [0]:
# 6. CREACIÓN DE TABLA USERS Y APLICACIÓN DE MASKING/RLS
catalog_name = "fintech_finpay"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog_name}.silver.users (
        user_id           STRING,
        full_name         STRING MASK {catalog_name}.silver.mask_pii,
        document_id       STRING MASK {catalog_name}.silver.mask_pii,
        email             STRING MASK {catalog_name}.silver.mask_pii,
        phone             STRING MASK {catalog_name}.silver.mask_pii,
        country           STRING,
        segment           STRING,
        registration_date DATE,
        _ingestion_date   TIMESTAMP
    )
    USING DELTA
    WITH ROW FILTER {catalog_name}.silver.row_filter_users ON (country)
    COMMENT 'Usuarios FinPay — PII protegido con Column Masking y RLS'
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print("✅ Tabla silver.users creada con políticas de seguridad activas.")

✅ Tabla silver.users creada con políticas de seguridad activas.
